# Notebook 1: Environment Setup, Data Generation & Feature Engineering

This notebook sets up the DEV/STAGING/PROD environment structure, generates realistic synthetic BNPL credit application data (500k applications, 20 features), and builds a Feature Store for the Probability of Default model.

**Pipeline mapping**: Replaces SageMaker Processing Step + S3 staging

**What this replaces in the current architecture:**
- AWS Organizations / separate accounts for DEV/STAGING/PROD --> Snowflake database-level isolation
- Terraform workspaces + IAM roles per environment --> RBAC roles with SQL GRANT statements
- S3 buckets for data staging between Snowflake and SageMaker --> eliminated entirely (data stays in Snowflake)
- SageMaker Processing Step for feature engineering --> Feature Store with Dynamic Tables (incremental refresh)

**Cost/performance advantage:** Zero data movement = zero S3 storage + transfer costs. Environment setup in minutes (SQL) vs hours (Terraform + IAM). Feature Store incremental refresh reprocesses only changed data, not full rebuilds.

## 1. Environment Setup (DEV / STAGING / PROD)

Replaces: Separate AWS accounts + Terraform workspaces + IAM roles

In [ ]:
-- =============================================================================
-- THREE-ENVIRONMENT INFRASTRUCTURE
-- Replaces: Separate AWS accounts + Terraform workspaces + IAM roles
-- Cost saving: No AWS Organizations overhead, no Terraform state management
-- Time saving: Minutes to set up vs hours of Terraform plan/apply cycles
-- =============================================================================

-- DEV: Where data scientists experiment and train
CREATE DATABASE IF NOT EXISTS PD_DEMO_DEV;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_DEV.ML;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_DEV.FEATURE_STORE;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_DEV.REGISTRY;

-- STAGING: Model validation and integration testing
CREATE DATABASE IF NOT EXISTS PD_DEMO_STAGING;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_STAGING.REGISTRY;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_STAGING.SERVING;

-- PROD: Live inference, monitoring, alerting
CREATE DATABASE IF NOT EXISTS PD_DEMO_PROD;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_PROD.REGISTRY;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_PROD.SERVING;
CREATE SCHEMA IF NOT EXISTS PD_DEMO_PROD.MONITORING;

-- Warehouse with AUTO_SUSPEND: only pay for compute when active
-- SageMaker notebook instances run 24/7 unless manually stopped
CREATE WAREHOUSE IF NOT EXISTS PD_DEMO_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE DATABASE PD_DEMO_DEV;
USE SCHEMA ML;
USE WAREHOUSE PD_DEMO_WH;

In [ ]:
-- =============================================================================
-- RBAC ROLES: Environment access control
-- Replaces: IAM policies across multiple AWS accounts + STS assume-role chains
-- Benefit: Single identity plane, no cross-account role assumption needed
-- =============================================================================
CREATE ROLE IF NOT EXISTS PD_DS_DEV;
CREATE ROLE IF NOT EXISTS PD_MLOPS;

-- DS team: full DEV access, read-only STAGING/PROD (cannot promote to prod)
GRANT USAGE ON DATABASE PD_DEMO_DEV TO ROLE PD_DS_DEV;
GRANT USAGE ON ALL SCHEMAS IN DATABASE PD_DEMO_DEV TO ROLE PD_DS_DEV;
GRANT ALL PRIVILEGES ON ALL SCHEMAS IN DATABASE PD_DEMO_DEV TO ROLE PD_DS_DEV;
GRANT USAGE ON DATABASE PD_DEMO_STAGING TO ROLE PD_DS_DEV;
GRANT USAGE ON DATABASE PD_DEMO_PROD TO ROLE PD_DS_DEV;

-- ML Ops team: full STAGING/PROD, can promote models and deploy endpoints
GRANT ALL PRIVILEGES ON DATABASE PD_DEMO_STAGING TO ROLE PD_MLOPS;
GRANT ALL PRIVILEGES ON ALL SCHEMAS IN DATABASE PD_DEMO_STAGING TO ROLE PD_MLOPS;
GRANT ALL PRIVILEGES ON DATABASE PD_DEMO_PROD TO ROLE PD_MLOPS;
GRANT ALL PRIVILEGES ON ALL SCHEMAS IN DATABASE PD_DEMO_PROD TO ROLE PD_MLOPS;
GRANT USAGE ON DATABASE PD_DEMO_DEV TO ROLE PD_MLOPS;

GRANT USAGE ON WAREHOUSE PD_DEMO_WH TO ROLE PD_DS_DEV;
GRANT USAGE ON WAREHOUSE PD_DEMO_WH TO ROLE PD_MLOPS;

## 2. Synthetic Data Generation

Generate realistic BNPL credit application data with credit bureau features. In production, this data comes from your credit bureau provider and is already in Snowflake -- eliminating the S3 staging step entirely.

In [ ]:
-- =============================================================================
-- SYNTHETIC CREDIT APPLICATION DATA: 500k applications, 20 features
-- In production, credit bureau data is already in Snowflake -- no S3 export needed
-- This eliminates the Snowflake -> S3 -> SageMaker data movement entirely
-- Cost saving: zero S3 storage, zero data transfer fees, zero ETL pipeline maintenance
-- =============================================================================
CREATE OR REPLACE TABLE PD_DEMO_DEV.ML.CREDIT_APPLICATIONS AS
WITH base AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY SEQ4()) AS row_id,
        'APP-' || LPAD(ROW_NUMBER() OVER (ORDER BY SEQ4())::STRING, 8, '0') AS application_id,
        'CUST-' || LPAD(UNIFORM(1, 400000, RANDOM())::STRING, 8, '0') AS customer_id,
        DATEADD('day', -UNIFORM(0, 730, RANDOM()), CURRENT_DATE()) AS application_date,
        -- Credit bureau features (17 numeric features)
        GREATEST(0, ROUND(NORMAL(4, 2.5, RANDOM()))) AS num_credit_products,
        GREATEST(0, ROUND(NORMAL(1.5, 1.2, RANDOM()))) AS num_inactive_accounts,
        GREATEST(0, ROUND(NORMAL(3, 2, RANDOM()))) AS num_credit_searches_l6m,
        GREATEST(0, ROUND(NORMAL(8500, 6000, RANDOM()), 2)) AS total_outstanding_balance,
        GREATEST(0, ROUND(NORMAL(5, 15, RANDOM()))) AS max_delinquency_days,
        LEAST(1.0, GREATEST(0, ROUND(NORMAL(0.35, 0.2, RANDOM()), 3))) AS credit_utilisation_ratio,
        GREATEST(6, ROUND(NORMAL(84, 48, RANDOM()))) AS months_since_oldest_account,
        GREATEST(0, ROUND(NORMAL(0.3, 0.7, RANDOM()))) AS num_defaults_l3y,
        GREATEST(0, ROUND(NORMAL(0.1, 0.4, RANDOM()))) AS num_ccjs,
        GREATEST(300, LEAST(850, ROUND(NORMAL(650, 80, RANDOM())))) AS credit_score,
        -- 7 additional bureau/application features
        GREATEST(0, ROUND(NORMAL(3, 2, RANDOM()))) AS num_open_accounts,
        GREATEST(0, ROUND(NORMAL(15000, 10000, RANDOM()), 2)) AS total_credit_limit,
        GREATEST(0, ROUND(NORMAL(0.5, 1.2, RANDOM()))) AS num_missed_payments_l12m,
        LEAST(2.0, GREATEST(0, ROUND(NORMAL(0.35, 0.2, RANDOM()), 3))) AS debt_to_income_ratio,
        GREATEST(0, ROUND(NORMAL(36, 30, RANDOM()))) AS months_since_last_default,
        GREATEST(0, ROUND(NORMAL(1.5, 1.5, RANDOM()))) AS num_hard_searches_l3m,
        GREATEST(18, LEAST(75, ROUND(NORMAL(32, 8, RANDOM())))) AS applicant_age_years,
        -- Origination channel (3 one-hot features after encoding = 20 model features total)
        CASE
            WHEN UNIFORM(1, 100, RANDOM()) <= 40 THEN 'Direct'
            WHEN UNIFORM(1, 100, RANDOM()) <= 75 THEN 'Google'
            ELSE 'Meta'
        END AS origination_channel,
        UNIFORM(0.0, 1.0, RANDOM()) AS rand_val
    FROM TABLE(GENERATOR(ROWCOUNT => 500000))
)
SELECT
    application_id, customer_id, application_date,
    num_credit_products, num_inactive_accounts, num_credit_searches_l6m,
    total_outstanding_balance, max_delinquency_days, credit_utilisation_ratio,
    months_since_oldest_account, num_defaults_l3y, num_ccjs, credit_score,
    num_open_accounts, total_credit_limit, num_missed_payments_l12m,
    debt_to_income_ratio, months_since_last_default, num_hard_searches_l3m,
    applicant_age_years, origination_channel,
    -- Target: ~6% default rate, correlated with risk factors
    CASE
        WHEN (
            0.10 * (num_credit_searches_l6m / 10.0)
            + 0.15 * (max_delinquency_days / 60.0)
            + 0.10 * credit_utilisation_ratio
            + 0.10 * (num_defaults_l3y / 3.0)
            + 0.05 * (num_ccjs / 2.0)
            + 0.15 * (1.0 - (credit_score - 300.0) / 550.0)
            + 0.05 * (num_missed_payments_l12m / 5.0)
            + 0.10 * LEAST(1.0, debt_to_income_ratio)
            + 0.05 * (num_hard_searches_l3m / 5.0)
            + 0.05 * (1.0 - LEAST(1.0, months_since_last_default / 60.0))
            + 0.10 * (1.0 - LEAST(1.0, applicant_age_years / 50.0))
        ) + (rand_val - 0.5) * 0.3 > 0.45
        THEN 1 ELSE 0
    END AS default_90dpm
FROM base;

In [ ]:
SELECT
    COUNT(*) AS total_applications,
    SUM(default_90dpm) AS total_defaults,
    ROUND(AVG(default_90dpm) * 100, 2) AS default_rate_pct,
    COUNT(DISTINCT origination_channel) AS num_channels
FROM PD_DEMO_DEV.ML.CREDIT_APPLICATIONS;

## 3. Data Quality Exploration

In [ ]:
# Data stays in Snowflake -- no S3 export needed for exploration
# SageMaker requires: Snowflake -> S3 -> SageMaker Notebook (data movement + cost)
# Snowflake: query directly from the notebook (zero data movement)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark.context import get_active_session

session = get_active_session()
df = session.table('PD_DEMO_DEV.ML.CREDIT_APPLICATIONS').to_pandas()

numeric_cols = [
    'NUM_CREDIT_PRODUCTS', 'NUM_INACTIVE_ACCOUNTS', 'NUM_CREDIT_SEARCHES_L6M',
    'TOTAL_OUTSTANDING_BALANCE', 'MAX_DELINQUENCY_DAYS', 'CREDIT_UTILISATION_RATIO',
    'MONTHS_SINCE_OLDEST_ACCOUNT', 'NUM_DEFAULTS_L3Y', 'NUM_CCJS', 'CREDIT_SCORE',
    'NUM_OPEN_ACCOUNTS', 'TOTAL_CREDIT_LIMIT', 'NUM_MISSED_PAYMENTS_L12M',
    'DEBT_TO_INCOME_RATIO', 'MONTHS_SINCE_LAST_DEFAULT', 'NUM_HARD_SEARCHES_L3M',
    'APPLICANT_AGE_YEARS'
]

print(f"Dataset shape: {df.shape}")
print(f"Unique customers: {df['CUSTOMER_ID'].nunique():,}")
print(f"\nDefault rate: {df['DEFAULT_90DPM'].mean():.2%}")
print(f"\nMissing values:\n{df[numeric_cols].isnull().sum()}")
df[numeric_cols].describe().round(2)

In [ ]:
# Visualise all 17 numeric features split by default status
# In SageMaker, you'd need to export data to S3 first to plot in a notebook
fig, axes = plt.subplots(3, 6, figsize=(24, 12))
fig.suptitle('Feature Distributions by Default Status (17 Bureau/Application Features)', fontsize=14)

for idx, c in enumerate(numeric_cols):
    ax = axes[idx // 6, idx % 6]
    for label, group in df.groupby('DEFAULT_90DPM'):
        ax.hist(group[c], bins=30, alpha=0.5, label=f'Default={label}', density=True)
    ax.set_title(c.replace('_', ' ').title(), fontsize=8)
    ax.legend(fontsize=6)

# Hide unused subplot
axes[2, 5].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr = df[numeric_cols + ['DEFAULT_90DPM']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Feature Correlation Matrix')

channel_defaults = df.groupby('ORIGINATION_CHANNEL')['DEFAULT_90DPM'].agg(['mean', 'count'])
channel_defaults['mean'].plot(kind='bar', ax=axes[1], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Default Rate by Origination Channel')
axes[1].set_ylabel('Default Rate')
axes[1].tick_params(axis='x', rotation=0)
for i, (rate, count) in enumerate(zip(channel_defaults['mean'], channel_defaults['count'])):
    axes[1].annotate(f'{rate:.2%}\n(n={count:,})', xy=(i, rate), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Feature Store Setup

The Snowflake Feature Store replaces the SageMaker Processing Step. Features are managed as Dynamic Tables with incremental refresh -- no S3 staging required.

**Key advantage**: Point-in-time correctness via ASOF JOIN prevents data leakage in training data generation -- critical for regulatory compliance.

In [ ]:
# =============================================================================
# FEATURE STORE: Replaces SageMaker Processing Step + S3 staging
# A Feature Store in Snowflake is simply a schema -- no external infrastructure.
# Feature views are backed by Dynamic Tables with incremental refresh:
#   - Only changed/new data is reprocessed (vs SageMaker full Processing Step reruns)
#   - Cost optimisation: pay only for incremental compute, not full table scans
# =============================================================================
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(
    session=session,
    database="PD_DEMO_DEV",
    name="FEATURE_STORE",
    default_warehouse="PD_DEMO_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="Credit application customer entity"
)
fs.register_entity(customer_entity)
print(f"Entity registered: {customer_entity.name}")
print(f"Registered entities: {fs.list_entities().to_pandas()['NAME'].tolist()}")

In [ ]:
# Bureau Feature View: all 17 numeric credit bureau + application features
# Backed by a Dynamic Table with incremental refresh -- only new/changed rows reprocessed
# Cost advantage: SageMaker Processing Step reruns the full dataset every time
from snowflake.snowpark.functions import col, when, lit

source_df = session.table('PD_DEMO_DEV.ML.CREDIT_APPLICATIONS')

bureau_df = source_df.select(
    col('CUSTOMER_ID'),
    col('APPLICATION_DATE').alias('EVENT_TS'),
    col('NUM_CREDIT_PRODUCTS'), col('NUM_INACTIVE_ACCOUNTS'),
    col('NUM_CREDIT_SEARCHES_L6M'), col('TOTAL_OUTSTANDING_BALANCE'),
    col('MAX_DELINQUENCY_DAYS'), col('CREDIT_UTILISATION_RATIO'),
    col('MONTHS_SINCE_OLDEST_ACCOUNT'), col('NUM_DEFAULTS_L3Y'),
    col('NUM_CCJS'), col('CREDIT_SCORE'),
    col('NUM_OPEN_ACCOUNTS'), col('TOTAL_CREDIT_LIMIT'),
    col('NUM_MISSED_PAYMENTS_L12M'), col('DEBT_TO_INCOME_RATIO'),
    col('MONTHS_SINCE_LAST_DEFAULT'), col('NUM_HARD_SEARCHES_L3M'),
    col('APPLICANT_AGE_YEARS')
)

bureau_fv = FeatureView(
    name="BUREAU_FEATURES",
    entities=[customer_entity],
    feature_df=bureau_df,
    timestamp_col="EVENT_TS",
    refresh_freq="60 minutes",
    desc="Credit bureau + application features for PD model (17 numeric features)"
)

bureau_fv = fs.register_feature_view(feature_view=bureau_fv, version="v1", block=True)
print(f"Bureau Feature View registered: {bureau_fv.name} v{bureau_fv.version}")
print(f"Features: {bureau_fv.feature_names}")

In [ ]:
# Channel Feature View: categorical encoding handled natively in Snowflake
# No separate preprocessing pipeline needed -- the encoding IS the feature view
# In SageMaker, you'd need a sklearn ColumnTransformer or Processing Step for this
channel_df = source_df.select(
    col('CUSTOMER_ID'),
    col('APPLICATION_DATE').alias('EVENT_TS'),
    col('ORIGINATION_CHANNEL'),
    when(col('ORIGINATION_CHANNEL') == 'Direct', 1).otherwise(0).alias('CHANNEL_DIRECT'),
    when(col('ORIGINATION_CHANNEL') == 'Google', 1).otherwise(0).alias('CHANNEL_GOOGLE'),
    when(col('ORIGINATION_CHANNEL') == 'Meta', 1).otherwise(0).alias('CHANNEL_META')
)

channel_fv = FeatureView(
    name="CHANNEL_FEATURES",
    entities=[customer_entity],
    feature_df=channel_df,
    timestamp_col="EVENT_TS",
    refresh_freq="60 minutes",
    desc="Origination channel one-hot encoded features"
)

channel_fv = fs.register_feature_view(feature_view=channel_fv, version="v1", block=True)
print(f"Channel Feature View registered: {channel_fv.name} v{channel_fv.version}")
print(f"\nAll feature views:")
print(fs.list_feature_views().to_pandas()[['NAME', 'VERSION', 'DESC']].to_string(index=False))

## 5. Preprocessing & Training Dataset Generation

The Feature Store's `generate_training_set` uses ASOF JOIN to ensure no future data leaks into the training set. In SageMaker, this is typically handled manually and is error-prone.

In [ ]:
# =============================================================================
# TRAINING DATASET with point-in-time correctness (ASOF JOIN)
# This is a critical regulatory advantage for credit risk:
#   - Prevents future data leaking into training (data leakage)
#   - In SageMaker, PIT correctness is handled manually and is error-prone
#   - Snowflake Feature Store enforces it by design
# Performance: runs as a Snowflake join operation, not a separate ETL job
# =============================================================================
spine_df = session.table('PD_DEMO_DEV.ML.CREDIT_APPLICATIONS').select(
    col('CUSTOMER_ID'),
    col('APPLICATION_DATE').alias('EVENT_TS'),
    col('DEFAULT_90DPM')
)

bureau_fv_ref = fs.get_feature_view('BUREAU_FEATURES', 'v1')
channel_fv_ref = fs.get_feature_view('CHANNEL_FEATURES', 'v1')

training_set = fs.generate_training_set(
    spine_df=spine_df,
    features=[bureau_fv_ref, channel_fv_ref],
    timestamp_col='EVENT_TS',
    spine_label_cols=['DEFAULT_90DPM']
)

training_pd = training_set.to_pandas()
print(f"Training set shape: {training_pd.shape}")
print(f"Features: {len([c for c in training_pd.columns if c not in ['CUSTOMER_ID','EVENT_TS','DEFAULT_90DPM']])} model features")
print(f"\nDefault rate in training set: {training_pd['DEFAULT_90DPM'].mean():.2%}")
training_pd.head()

In [ ]:
training_set.write.mode('overwrite').save_as_table('PD_DEMO_DEV.ML.TRAINING_DATA')
print(f"Training data saved to PD_DEMO_DEV.ML.TRAINING_DATA")
print(f"Rows: {session.table('PD_DEMO_DEV.ML.TRAINING_DATA').count():,}")

---
## Summary

| What we did | SageMaker equivalent | Snowflake approach | Cost/perf advantage |
|---|---|---|---|
| Created 3 environments | Terraform + AWS Orgs + IAM (hours) | ~20 SQL statements (minutes) | No Terraform state, no IAM overhead |
| Generated 500k applications | S3 + SageMaker Processing Step | 1 SQL query (data stays in Snowflake) | Zero data transfer cost |
| Built Feature Store (20 features) | S3 + custom feature pipeline | Feature Store with Dynamic Tables | Incremental refresh (only changed data) |
| Point-in-time training dataset | Manual timestamp handling (error-prone) | 1 `generate_training_set` with ASOF JOIN | Regulatory compliance by design |

**Next**: Notebook 2 -- Model Training, Hyperparameter Tuning, and Environment Promotion